In [1]:
import chromadb
import os

os.chdir("..")
print(os.getcwd())

client = chromadb.PersistentClient(path="chromadb_context")
collection = client.get_collection(name="document_collection")

print(f"Collection loaded: {collection.count()} chunks indexed")

c:\Users\manx1\Documents\LLM_RAG_Agents
Collection loaded: 749 chunks indexed


In [2]:
test_set = [
    {
        "question": "How does importance sampling improve PINN training efficiency?",
        "expected_source": "Efficient_training_of_PINN_via_Importance_Sampling.pdf"
    },
    {
        "question": "What are the applications of PINNs in fluid mechanics?",
        "expected_source": "PINN_for_fluid_mechanics_review.pdf"
    },
    {
        "question": "How are PINNs used as surrogate models in hydrodynamics?",
        "expected_source": "PINN_as_surrogate_ML_for_hydrodynamics.pdf"
    },
    {
        "question": "Where and what are the current applications of machine learning through PINNs?",
        "expected_source": "ML_through_PINN_where_and_what.pdf"
    },
    {
        "question": "What is physics informed machine learning?",
        "expected_source": "physics_informed_ml.pdf"
    },
    {
    "question": "What is the capital of France?",
    "expected_source": None
    }
]

print(f"Test set ready: {len(test_set)} questions")

Test set ready: 6 questions


In [3]:
def evaluate_retrieval(test_set, distance_threshold=1.3):
    hit_count = 0
    for i in test_set:
        question = i["question"]
        expected_source = i["expected_source"]
        results = collection.query(
            query_texts=[question],
            n_results=3,
            include=["distances"]
        )
        ids = results["ids"][0]
        distances = results["distances"][0]
        retrieved_filenames = [id.rsplit("_chunk_", 1)[0] for id in ids]

        if expected_source is None:
            if distances[0] > distance_threshold:
                hit_count += 1
                print(f"✅ CORRECTLY REJECTED — {question[:50]}...")
            else:
                print(f"❌ FALSE MATCH — {question[:50]}...")
                print(f"   Got: {retrieved_filenames[0]} (distance={distances[0]:.2f})")
        else:
            if expected_source in retrieved_filenames:
                hit_count += 1
                print(f"✅ HIT — {question[:50]}...")
            else:
                print(f"❌ MISS — {question[:50]}...")
                print(f"   Expected: {expected_source}, Got: {retrieved_filenames}")

    print(f"\nActual hits across test set: {hit_count*100/len(test_set)}%")

In [4]:
evaluate_retrieval(test_set)

✅ HIT — How does importance sampling improve PINN training...
✅ HIT — What are the applications of PINNs in fluid mechan...
✅ HIT — How are PINNs used as surrogate models in hydrodyn...
✅ HIT — Where and what are the current applications of mac...
✅ HIT — What is physics informed machine learning?...
✅ CORRECTLY REJECTED — What is the capital of France?...

Actual hits across test set: 100.0%


In [5]:
import re

def strip_thinking(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

In [ ]:
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
groq_client = Groq()

def check_faithfulness(question, retrieved_chunks, answer):
    context = "\n\n".join(retrieved_chunks)

    judge_prompt = f"""You are a strict fact-checker. Given the CONTEXT and an ANSWER, determine if the ANSWER is fully supported by the CONTEXT.

CONTEXT:
{context}

QUESTION: {question}

ANSWER: {answer}

Respond with only one word: FAITHFUL if every claim in the answer is supported by the context, or UNFAITHFUL if the answer contains information not found in the context."""

    response = groq_client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[{"role": "user", "content": judge_prompt}],
        max_tokens=700
    )

    verdict = strip_thinking(response.choices[0].message.content)
    return verdict

In [9]:
test_question = test_set[0]["question"]

results = collection.query(query_texts=[test_question], n_results=3)
chunks = results["documents"][0]

answer_response = groq_client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[{"role": "user", "content": f"Context:\n{chr(10).join(chunks)}\n\nQuestion: {test_question}\n\nAnswer using only the context above."}],
    max_tokens=2400
)
answer = strip_thinking(answer_response.choices[0].message.content)

verdict = check_faithfulness(test_question, chunks, answer)
print(f"Question: {test_question}\n")
print(f"Answer: {answer}\n")
print(f"Verdict: {verdict}")

Question: How does importance sampling improve PINN training efficiency?

Answer: The importance sampling method improves PINN training efficiency by adaptively focusing on regions with higher loss values, leading to faster convergence. Key steps include:  
1. **Seed-based Loss Estimation**: Computing loss values at predefined "seed" points to guide sampling.  
2. **Dynamic Sampling**: Assigning probabilities proportional to the loss magnitude at each collocation point (computed via $ \tilde{q}_j = J(\theta;x_{\rho(j)}) / \sum_j J(\theta;x_{\rho(j)}) $), ensuring more samples are drawn from regions where the model performs poorly.  
3. **Reduced Iterations and Time**: As demonstrated in the context, this approach reduces training iterations and computational time compared to uniform sampling and exact/PWC loss evaluations (e.g., 40 seconds for PINNs vs. 32 for FEM).  
4. **Computational Validation**: Figures in the context show that importance sampling accelerates convergence by concen

In [ ]:
for i in test_set:
    if i["expected_source"] is None:
        continue

    question = i["question"]
    results = collection.query(query_texts=[question], n_results=3)
    chunks = results["documents"][0]

    answer_response = groq_client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[{"role": "user", "content": f"Context:\n{chr(10).join(chunks)}\n\nQuestion: {question}\n\nAnswer using only the context above."}],
        max_tokens=1200
    )
    answer = strip_thinking(answer_response.choices[0].message.content)
    verdict = strip_thinking(check_faithfulness(question, chunks, answer))

    print(f"{verdict} — {question[:60]}...")

FAITHFUL — How does importance sampling improve PINN training efficienc...
FAITHFUL — What are the applications of PINNs in fluid mechanics?...
FAITHFUL — How are PINNs used as surrogate models in hydrodynamics?...
<think>
Okay, let's tackle this. The user wants to check if the provided answer about PINNs is fully supported by the given context.

First, I need to go through the answer and cross-reference each part with the context. The answer mentions applications of PINNs like solving PDEs in computational physics and engineering, and specifically mentions solving known physical models from Raissi's research. The context does talk about Raissi's 2019 work and using PINNs for known models. So that part seems okay.

Next, the answer states that PINNs use physical information in loss functions without labeled data. The context supports this, saying they don't require labeled data and use governing equations. Then, it mentions boundary conditions, initial value problems, and collocation p